# 01 — Explorare & curățare populație (INS TEMPO)

Scop: din 3 (apoi 42) CSV-uri brute → un singur tabel curat, o linie per localitate.

## 1. Citește toate fișierele din `data/raw/populatie/` cu `glob`

In [1]:
import glob
import pandas as pd

def read_csv_files(file_pattern):
    """
    Reads multiple CSV files matching the given file pattern and concatenates them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files (e.g., 'data/*.csv').

    Returns:
    pd.DataFrame: A DataFrame containing the concatenated data from all matched CSV files.
    """
    csv_files = sorted(glob.glob(file_pattern))
    if not csv_files:
        raise FileNotFoundError()

    # Initialize an empty list to hold DataFrames
    dataframes = []
    
    # Loop through the list of CSV files and read each one into a DataFrame
    for file in csv_files:
        df = pd.read_csv(file, skipinitialspace=True)
        dataframes.append(df)
    
    # Concatenate all DataFrames into a single DataFrame
    combined_df = pd.concat(dataframes, ignore_index=True)
    
    return combined_df

## 2. Concatenează într-un singur DataFrame

In [2]:
df = read_csv_files("../data/raw/populatie/*.csv")


In [3]:
df.head(10)

,Varste si grupe de varsta,Sexe,Judete,Localitati,Ani,UM: Numar persoane,Valoare
0,Total,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,2108049
1,65-69 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,126370
2,70-74 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,122788
3,75-79 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,81809
4,80-84 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,41639
5,85 ani si peste,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,53995
6,Total,Total,Alba,1017 MUNICIPIUL ALBA IULIA,Anul 2026,Numar persoane,74137
7,Total,Total,Alba,1213 MUNICIPIUL AIUD,Anul 2026,Numar persoane,23600
8,Total,Total,Alba,1348 MUNICIPIUL BLAJ,Anul 2026,Numar persoane,19821
9,Total,Total,Alba,1874 MUNICIPIUL SEBES,Anul 2026,Numar persoane,31880


In [4]:
df.shape

(19086, 7)

In [5]:
df.columns.tolist()

['Varste si grupe de varsta',
 'Sexe',
 'Judete',
 'Localitati ',
 'Ani',
 'UM: Numar persoane',
 'Valoare']

## 3. Curăță numele coloanelor

Atenție: `'Localitati '` are spațiu la final.

In [6]:
df.columns = df.columns.str.strip()

In [7]:
df

,Varste si grupe de varsta,Sexe,Judete,Localitati,Ani,UM: Numar persoane,Valoare
0,Total,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,2108049
1,65-69 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,126370
2,70-74 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,122788
3,75-79 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,81809
4,80-84 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,41639
...,...,...,...,...,...,...,...
19081,85 ani si peste,Total,Vrancea,178377 VIDRA,Anul 2026,Numar persoane,186
19082,85 ani si peste,Total,Vrancea,178475 VINTILEASCA,Anul 2026,Numar persoane,62
19083,85 ani si peste,Total,Vrancea,178545 VIZANTEA-LIVEZI,Anul 2026,Numar persoane,123
19084,85 ani si peste,Total,Vrancea,178750 VRANCIOAIA,Anul 2026,Numar persoane,128


## 4. Separă codul SIRUTA de numele localității

`2130 ALBAC` vs `1017 MUNICIPIUL ALBA IULIA` — care e regula care merge în ambele cazuri?

In [8]:
df[['cod_siruta', 'localitate']] = df["Localitati"].str.split(" ", n=1, expand=True)

In [9]:
df.head()

,Varste si grupe de varsta,Sexe,Judete,Localitati,Ani,UM: Numar persoane,Valoare,cod_siruta,localitate
0,Total,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,2108049,179132,MUNICIPIUL BUCURESTI
1,65-69 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,126370,179132,MUNICIPIUL BUCURESTI
2,70-74 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,122788,179132,MUNICIPIUL BUCURESTI
3,75-79 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,81809,179132,MUNICIPIUL BUCURESTI
4,80-84 ani,Total,Municipiul Bucuresti,179132 MUNICIPIUL BUCURESTI,Anul 2026,Numar persoane,41639,179132,MUNICIPIUL BUCURESTI


In [10]:
df.dtypes

Varste si grupe de varsta      str
Sexe                           str
Judete                         str
Localitati                     str
Ani                            str
UM: Numar persoane             str
Valoare                      int64
cod_siruta                     str
localitate                     str
dtype: object

In [11]:
df.columns.tolist()


['Varste si grupe de varsta',
 'Sexe',
 'Judete',
 'Localitati',
 'Ani',
 'UM: Numar persoane',
 'Valoare',
 'cod_siruta',
 'localitate']

## 5. Pivotează: din format lung în format lat

O linie per localitate, câte o coloană per grupă de vârstă.

In [12]:
df_lat = df.pivot(index=["Judete",'cod_siruta','localitate'], columns="Varste si grupe de varsta", values="Valoare")

In [13]:
df_lat

Varste si grupe de varsta                 65-69 ani  70-74 ani  75-79 ani  \
Judete  cod_siruta localitate                                               
Alba    1017       MUNICIPIUL ALBA IULIA       5082       4639       2698   
        1071       CIUGUD                       179        208        136   
        1151       ORAS ABRUD                   331        296        179   
        1213       MUNICIPIUL AIUD             1736       1686       1022   
        1348       MUNICIPIUL BLAJ             1239       1221        824   
...                                             ...        ...        ...   
Vrancea 178929     BILIESTI                     141        131         92   
        178938     GOLOGANU                     207        159        149   
        178947     OBREJITA                      77         77         62   
        178956     RASTOACA                     156        139        118   
        178965     SPULBER                       50         55         52   

Varste si grupe de varsta                 80-84 ani  85 ani si peste  Total  
Judete  cod_siruta localitate                                                
Alba    1017       MUNICIPIUL ALBA IULIA       1227              994  74137  
        1071       CIUGUD                        74               54   3448  
        1151       ORAS ABRUD                   110               87   4829  
        1213       MUNICIPIUL AIUD              510              477  23600  
        1348       MUNICIPIUL BLAJ              454              359  19821  
...                                             ...              ...    ...  
Vrancea 178929     BILIESTI                      56               63   2434  
        178938     GOLOGANU                     109              116   2833  
        178947     OBREJITA                      37               42   1542  
        178956     RASTOACA                      79               89   2259  
        178965     SPULBER                       24               42   1267  

[3181 rows x 6 columns]

In [14]:
df_lat.shape

(3181, 6)

In [15]:
df_lat.columns.tolist()

['65-69 ani',
 '70-74 ani',
 '75-79 ani',
 '80-84 ani',
 '85 ani si peste',
 'Total']

In [16]:
df_lat = df_lat.reset_index()

## 6. Coloane derivate: `populatie_65plus` și `pondere_65plus`

In [17]:
df_lat['populatie_65plus'] = df_lat[['65-69 ani', '70-74 ani', '75-79 ani', '80-84 ani', '85 ani si peste']].sum(axis=1)

In [18]:
df_lat

Varste si grupe de varsta,Judete,cod_siruta,localitate,65-69 ani,70-74 ani,75-79 ani,80-84 ani,85 ani si peste,Total,populatie_65plus
0,Alba,1017,MUNICIPIUL ALBA IULIA,5082,4639,2698,1227,994,74137,14640
1,Alba,1071,CIUGUD,179,208,136,74,54,3448,651
2,Alba,1151,ORAS ABRUD,331,296,179,110,87,4829,1003
3,Alba,1213,MUNICIPIUL AIUD,1736,1686,1022,510,477,23600,5431
4,Alba,1348,MUNICIPIUL BLAJ,1239,1221,824,454,359,19821,4097
...,...,...,...,...,...,...,...,...,...,...
3176,Vrancea,178929,BILIESTI,141,131,92,56,63,2434,483
3177,Vrancea,178938,GOLOGANU,207,159,149,109,116,2833,740
3178,Vrancea,178947,OBREJITA,77,77,62,37,42,1542,295
3179,Vrancea,178956,RASTOACA,156,139,118,79,89,2259,581


In [19]:
df_lat['pondere_65plus'] = df_lat['populatie_65plus'] / df_lat['Total']

In [20]:
df_lat

Varste si grupe de varsta,Judete,cod_siruta,localitate,65-69 ani,70-74 ani,75-79 ani,80-84 ani,85 ani si peste,Total,populatie_65plus,pondere_65plus
0,Alba,1017,MUNICIPIUL ALBA IULIA,5082,4639,2698,1227,994,74137,14640,0.197472
1,Alba,1071,CIUGUD,179,208,136,74,54,3448,651,0.188805
2,Alba,1151,ORAS ABRUD,331,296,179,110,87,4829,1003,0.207703
3,Alba,1213,MUNICIPIUL AIUD,1736,1686,1022,510,477,23600,5431,0.230127
4,Alba,1348,MUNICIPIUL BLAJ,1239,1221,824,454,359,19821,4097,0.206700
...,...,...,...,...,...,...,...,...,...,...,...
3176,Vrancea,178929,BILIESTI,141,131,92,56,63,2434,483,0.198439
3177,Vrancea,178938,GOLOGANU,207,159,149,109,116,2833,740,0.261207
3178,Vrancea,178947,OBREJITA,77,77,62,37,42,1542,295,0.191310
3179,Vrancea,178956,RASTOACA,156,139,118,79,89,2259,581,0.257193


## 7. Verificări

Câte localități am? Se potrivește suma grupelor cu ce ar trebui? Unde arată ponderea suspect?

In [21]:
# --- Verificări pe setul complet, ÎNAINTE de orice excludere ---
# Valorile așteptate sunt definite o singură dată: mesajul nu poate contrazice condiția.
N_LOCALITATI    = 3181
N_COLOANE       = 11
TOTAL_65PLUS    = 4_076_589
TOTAL_POPULATIE = 21_646_220

# 1. nu am pierdut și nu am inventat localități
assert len(df_lat) == N_LOCALITATI, \
    f"asteptam {N_LOCALITATI} localitati, am {len(df_lat)}"

# 2. structura tabelului e cea așteptată
assert len(df_lat.columns) == N_COLOANE, \
    f"asteptam {N_COLOANE} coloane, am {len(df_lat.columns)}: {df_lat.columns.tolist()}"

# 3. cheia de join e într-adevăr o cheie — dacă se strică, join-ul cu sursa
#    de spitale va multiplica rânduri în tăcere, nu va crăpa
dupl = df_lat["cod_siruta"].duplicated()
assert not dupl.any(), (
    f"{dupl.sum()} coduri SIRUTA duplicate: "
    f"{df_lat.loc[df_lat['cod_siruta'].duplicated(keep=False), 'cod_siruta'].unique()[:10].tolist()}"
)

# 4. nicio valoare lipsă — sum(axis=1) tratează NaN ca 0 și ar fi ascuns problema
lipsa = df_lat.isna().sum()
assert lipsa.sum() == 0, f"valori lipsa pe coloane:\n{lipsa[lipsa > 0].to_string()}"

# 5. invariant logic: o submulțime nu poate depăși întregul
peste = ~df_lat["populatie_65plus"].between(0, df_lat["Total"])
assert not peste.any(), (
    f"{peste.sum()} localitati cu populatie_65plus in afara intervalului [0, Total]:\n"
    f"{df_lat.loc[peste, ['localitate', 'populatie_65plus', 'Total']].head(10).to_string()}"
)

# 6. ponderea e un raport, deci în [0, 1]; prinde și eventualele împărțiri la zero (inf)
afara = ~df_lat["pondere_65plus"].between(0, 1)
assert not afara.any(), (
    f"{afara.sum()} localitati cu pondere_65plus in afara intervalului [0, 1]:\n"
    f"{df_lat.loc[afara, ['localitate', 'Total', 'pondere_65plus']].head(10).to_string()}"
)

# 7-8. verificare încrucișată: cifrele calculate independent cu awk, pe CSV-urile brute
assert df_lat["populatie_65plus"].sum() == TOTAL_65PLUS, \
    f"total 65+: asteptam {TOTAL_65PLUS:,}, am {df_lat['populatie_65plus'].sum():,}"

assert df_lat["Total"].sum() == TOTAL_POPULATIE, \
    f"populatie totala: asteptam {TOTAL_POPULATIE:,}, am {df_lat['Total'].sum():,}"

print(f"✓ {len(df_lat):,} localitati · {N_COLOANE} coloane · 8 verificari trecute")

✓ 3,181 localitati · 11 coloane · 8 verificari trecute


In [22]:
df_lucru = df_lat[df_lat['cod_siruta'] != "179132"]

In [23]:
df_lucru

Varste si grupe de varsta,Judete,cod_siruta,localitate,65-69 ani,70-74 ani,75-79 ani,80-84 ani,85 ani si peste,Total,populatie_65plus,pondere_65plus
0,Alba,1017,MUNICIPIUL ALBA IULIA,5082,4639,2698,1227,994,74137,14640,0.197472
1,Alba,1071,CIUGUD,179,208,136,74,54,3448,651,0.188805
2,Alba,1151,ORAS ABRUD,331,296,179,110,87,4829,1003,0.207703
3,Alba,1213,MUNICIPIUL AIUD,1736,1686,1022,510,477,23600,5431,0.230127
4,Alba,1348,MUNICIPIUL BLAJ,1239,1221,824,454,359,19821,4097,0.206700
...,...,...,...,...,...,...,...,...,...,...,...
3176,Vrancea,178929,BILIESTI,141,131,92,56,63,2434,483,0.198439
3177,Vrancea,178938,GOLOGANU,207,159,149,109,116,2833,740,0.261207
3178,Vrancea,178947,OBREJITA,77,77,62,37,42,1542,295,0.191310
3179,Vrancea,178956,RASTOACA,156,139,118,79,89,2259,581,0.257193


In [24]:
# Filtrul a eliminat exact un rând?
# Un filtru care nu filtrează nu dă eroare — lasă Bucureștiul (2,1M locuitori)
# în toate agregările și strică rezultatul în tăcere.
eliminate = len(df_lat) - len(df_lucru)
assert eliminate == 1, (
    f"filtrul trebuia sa elimine exact 1 rand (Bucuresti), a eliminat {eliminate}. "
    f"Verifica tipul si formatul lui cod_siruta: {df_lat['cod_siruta'].dtype}"
)

print(f"✓ {len(df_lat):,} -> {len(df_lucru):,} localitati "
      f"({df_lucru['Total'].sum():,} locuitori, "
      f"{df_lucru['populatie_65plus'].sum() / df_lucru['Total'].sum():.1%} de 65+)")

✓ 3,181 -> 3,180 localitati (19,538,171 locuitori, 18.7% de 65+)


## 8. Export

Rezultatul se salvează în `data/processed/`, ca pașii următori să nu depindă de rularea
acestui notebook.

In [25]:
from pathlib import Path

IESIRE = Path("../data/processed/populatie_localitati.csv")
IESIRE.parent.mkdir(parents=True, exist_ok=True)

df_lucru.to_csv(IESIRE, index=False)

# Verificare de dus-întors: ce am scris se citește înapoi identic?
# CSV-ul nu păstrează tipurile — cod_siruta ar fi recitit ca int64 și ar rupe
# join-ul cu sursa de spitale, în tăcere. De aceea îl forțăm explicit la citire.
verif = pd.read_csv(IESIRE, dtype={"cod_siruta": "str"})
assert len(verif) == len(df_lucru), f"scris {len(df_lucru)} randuri, recitit {len(verif)}"
assert verif["Total"].sum() == df_lucru["Total"].sum(), "totalurile nu se potrivesc dupa export"

print(f"✓ {IESIRE} — {len(verif):,} randuri, {IESIRE.stat().st_size / 1024:.0f} KB")

✓ ../data/processed/populatie_localitati.csv — 3,180 randuri, 219 KB
